# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [9]:
# TU CÓDIGO AQUÍ:
# Utiliza TransactionEncoder para crear df_cafe_bool
te = TransactionEncoder()
df_cafe_bool = te.fit_transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(df_cafe_bool, columns=te.columns_)
df_cafe_bool.head()

,Agua,Brownie,Cafe Americano,Croissant,Ensalada,Frappe Mocha,Galletas,Jugo de Naranja,Muffin,Te Verde
0,True,False,True,True,False,False,False,False,False,False
1,True,False,False,False,False,True,False,False,True,False
2,False,False,True,True,False,False,False,True,False,False
3,True,False,False,False,False,False,True,False,False,True
4,False,False,False,False,False,True,True,False,True,False


### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [6]:
# TU CÓDIGO AQUÍ:
# Usa la función apriori() pasándole tu df_cafe_bool. Guarda el resultado en itemsets_cafe
itemsets_cafe = apriori(df_cafe_bool, min_support=0.25, use_colnames=True)
print(itemsets_cafe)

   support                     itemsets
0      0.6                       (Agua)
1      0.3                    (Brownie)
2      0.4             (Cafe Americano)
3      0.3                  (Croissant)
4      0.4               (Frappe Mocha)
5      0.3                   (Galletas)
6      0.3                     (Muffin)
7      0.3  (Croissant, Cafe Americano)
8      0.3       (Frappe Mocha, Muffin)


### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [10]:
# TU CÓDIGO AQUÍ:
# Usa association_rules() y guarda el resultado en reglas_cafe
# Filtra las columnas para imprimir solo: antecedents, consequents, support, confidence, lift
reglas_cafe = association_rules(itemsets_cafe, metric="confidence", min_threshold=0.6)
reglas_cafe = reglas_cafe[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

print(reglas_cafe)


        antecedents       consequents  support  confidence  lift
0       (Croissant)  (Cafe Americano)      0.3        1.00   2.5
1  (Cafe Americano)       (Croissant)      0.3        0.75   2.5
2    (Frappe Mocha)          (Muffin)      0.3        0.75   2.5
3          (Muffin)    (Frappe Mocha)      0.3        1.00   2.5


### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z"). 
R: Existen 2 con un 100% de confianza, la primera de que es seguro de que si alguien compra un croissant va a comrpar un café Americano, y la de si alguien compra un Muffin va a comprar un Frappe Mocha
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué. 
R: Observando el dataset, podemos ver que el agua es un producto muy vendido, pero no existe correlación con las ventas de frappe.